## LiTS Tumor-First Segmentation Pipeline (Kaggle)

This notebook is rewritten for robust, reproducible training with **Tumor Dice as primary selection target**.

### What this notebook guarantees
- strict split usage (`train`, `val`, `test`)
- fail-fast data loading (no silent zero replacements)
- deterministic/reproducible run controls
- medically-consistent preprocessing for CT PNG slices
- threshold calibration on validation for tumor/liver channels
- final held-out test reporting with clear metrics exports

### Primary objective
Maximize **validation Tumor Dice** while preserving strong liver segmentation and stable generalization.

In [ ]:
# Cell 1 — Environment, Reproducibility, and Global Config
import os
import gc
import random
import json
import math
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# Kaggle-friendly CUDA allocator
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Preset selector: "balanced_best" (recommended) or "aggressive_max_accuracy"
PRESET_NAME = "balanced_best"  # Stage-1 default; switch to aggressive only after gate passes

CFG_PRESETS = {
    "balanced_best": {
        "seed": 42,
        "deterministic": False,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "num_workers": 2,
        "pin_memory": True,
        "image_size": 256,
        "batch_size": 16,
        "accum_steps": 2,
        "epochs": 120,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "grad_clip": 5.0,
        "warmup_epochs": 5,
        "early_stop_patience": 18,
        "lr_patience": 5,
        "lr_factor": 0.5,
        "tumor_pos_weight": 3.2,
        "sampler_tumor_boost": 5.0,
        "sampler_liver_boost": 3.0,
        "dice_smooth": 1e-6,
        "outside_liver_penalty": 0.15,
        "full_png_audit": False,
        "audit_sample_rows": 3000,
        "liver_min_dice_guard": 0.90,
        "max_tumor_fp_rate": 0.005,
        "max_tumor_outside_liver_rate": 0.03,
        "composite_weights": {"tumor": 0.52, "liver": 0.38, "fp_penalty": 0.06, "outside_penalty": 0.04},
        "auto_resume": True,
        "save_every_n_epochs": 1,
        "eval_every_n_epochs": 1,
        "smoke_train_steps": 20,
        "max_bad_epochs": 3,
        "nan_loss_abort_batches": 200,
        "save_dir": "results",
        "model_dir": "models",
        "max_run_hours": 11.5,
        "weekly_gpu_budget_hours": 30.0,
    },
    "aggressive_max_accuracy": {
        "seed": 42,
        "deterministic": False,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "num_workers": 2,
        "pin_memory": True,
        "image_size": 320,
        "batch_size": 10,
        "accum_steps": 4,
        "epochs": 180,
        "lr": 2e-4,
        "weight_decay": 8e-5,
        "grad_clip": 3.5,
        "warmup_epochs": 8,
        "early_stop_patience": 24,
        "lr_patience": 6,
        "lr_factor": 0.5,
        "tumor_pos_weight": 4.2,
        "sampler_tumor_boost": 7.5,
        "sampler_liver_boost": 3.2,
        "dice_smooth": 1e-6,
        "outside_liver_penalty": 0.12,
        "full_png_audit": False,
        "audit_sample_rows": 4000,
        "liver_min_dice_guard": 0.91,
        "max_tumor_fp_rate": 0.006,
        "max_tumor_outside_liver_rate": 0.03,
        "composite_weights": {"tumor": 0.60, "liver": 0.28, "fp_penalty": 0.08, "outside_penalty": 0.04},
        "auto_resume": True,
        "save_every_n_epochs": 1,
        "eval_every_n_epochs": 1,
        "smoke_train_steps": 30,
        "max_bad_epochs": 4,
        "nan_loss_abort_batches": 300,
        "save_dir": "results",
        "model_dir": "models",
        "max_run_hours": 11.5,
        "weekly_gpu_budget_hours": 30.0,
    },
}

assert PRESET_NAME in CFG_PRESETS, f"Unknown PRESET_NAME: {PRESET_NAME}"
CFG = dict(CFG_PRESETS[PRESET_NAME])
CFG["run_name"] = f"{PRESET_NAME}_seed{CFG['seed']}_{int(time.time())}"

# Stage transition gate: move to aggressive only if balanced model is strong and stable
STAGE_GATE = {
    "min_val_tumor_dice": 0.60,
    "min_test_tumor_dice": 0.56,
    "min_val_liver_dice": 0.90,
    "min_test_liver_dice": 0.90,
    "max_val_test_tumor_gap": 0.05,
    "max_test_tumor_fp_rate": 0.008,
    "max_test_tumor_outside_liver_rate": 0.03,
}

KAGGLE_DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/andrewmvd/lits-png/dataset_6"),
    Path("/kaggle/input/datasets/andrewmvd/lits-png/dataset_6/dataset_6"),
]
LOCAL_DATA_ROOT = Path("data/dataset_6")
CSV_ROOT = Path("data")

def pick_existing_path(candidates):
    for p in candidates:
        if p.exists():
            return p
    return None

kaggle_root = pick_existing_path(KAGGLE_DATA_ROOT_CANDIDATES)
PATHS = {
    "data_root": kaggle_root if kaggle_root is not None else LOCAL_DATA_ROOT,
    "train_csv": CSV_ROOT / "lits_train.csv",
    "val_csv": CSV_ROOT / "lits_val.csv",
    "test_csv": CSV_ROOT / "lits_test.csv",
    "probe_csv": CSV_ROOT / "lits_probe.csv",
    "all_csv": CSV_ROOT / "lits_df.csv",
}

Path(CFG["save_dir"]).mkdir(parents=True, exist_ok=True)
Path(CFG["model_dir"]).mkdir(parents=True, exist_ok=True)

def seed_everything(seed: int = 42, deterministic: bool = False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

seed_everything(CFG["seed"], CFG["deterministic"])
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Device: {CFG['device']}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Data root: {PATHS['data_root']}")
print(f"Run name: {CFG['run_name']}")

## 0) Kaggle Dataset Path Override (Edit This First)

If Kaggle mounts your dataset at a different path, set it here once.
Use absolute Kaggle paths (example: `/kaggle/input/your-dataset-name/...`).

In [ ]:
# Cell 2 — Set dataset roots (paste Kaggle paths here if needed)
# Example:
# DATA_ROOT_OVERRIDE = "/kaggle/input/datasets/andrewmvd/lits-png/dataset_6"
# (if needed, this can also be .../dataset_6/dataset_6; notebook auto-resolves)
# CSV_ROOT_OVERRIDE  = "/kaggle/input/datasets/andrewmvd/lits-png"
DATA_ROOT_OVERRIDE = ""
CSV_ROOT_OVERRIDE = ""
USE_PROBE_FOR_ANALYSIS = False  # Keep False for this project

if DATA_ROOT_OVERRIDE.strip():
    PATHS["data_root"] = Path(DATA_ROOT_OVERRIDE.strip())
if CSV_ROOT_OVERRIDE.strip():
    csv_root_override = Path(CSV_ROOT_OVERRIDE.strip())
    PATHS["train_csv"] = csv_root_override / "lits_train.csv"
    PATHS["val_csv"] = csv_root_override / "lits_val.csv"
    PATHS["test_csv"] = csv_root_override / "lits_test.csv"
    PATHS["probe_csv"] = csv_root_override / "lits_probe.csv"
    PATHS["all_csv"] = csv_root_override / "lits_df.csv"

print("Using paths:")
for k, v in PATHS.items():
    print(f"  {k}: {v}")

assert PATHS["data_root"].exists(), f"Data root not found: {PATHS['data_root']}"
for key in ["train_csv", "val_csv", "test_csv", "all_csv"]:
    assert PATHS[key].exists(), f"Missing {key}: {PATHS[key]}"

# Validate split CSVs, schema, and leakage risk
required_cols = [
    "filepath", "liver_maskpath", "tumor_maskpath",
    "study_number", "instance_number", "liver_mask_empty", "tumor_mask_empty"
]

split_entries = [
    ("train", PATHS["train_csv"]),
    ("val", PATHS["val_csv"]),
    ("test", PATHS["test_csv"]),
    ("all", PATHS["all_csv"]),
]
if USE_PROBE_FOR_ANALYSIS and PATHS["probe_csv"].exists():
    split_entries.append(("probe", PATHS["probe_csv"]))

splits = {}
for split_name, csv_path in split_entries:
    df = pd.read_csv(csv_path)
    miss = [c for c in required_cols if c not in df.columns]
    assert not miss, f"{split_name} missing columns: {miss}"
    splits[split_name] = df

print("Rows:")
for k in ["train", "val", "test", "all"]:
    print(f"  {k:>5}: {len(splits[k]):,}")
if "probe" in splits:
    print(f"  {'probe':>5}: {len(splits['probe']):,}")

print("\nCardinality check:")
combined = len(splits["train"]) + len(splits["val"]) + len(splits["test"])
print(f"  train+val+test = {combined:,}")
print(f"  all            = {len(splits['all']):,}")

train_studies = set(splits["train"]["study_number"].astype(int).tolist())
val_studies = set(splits["val"]["study_number"].astype(int).tolist())
test_studies = set(splits["test"]["study_number"].astype(int).tolist())
print("\nStudy overlap:")
print(f"  train∩val : {len(train_studies & val_studies)}")
print(f"  train∩test: {len(train_studies & test_studies)}")
print(f"  val∩test  : {len(val_studies & test_studies)}")
assert len(train_studies & val_studies) == 0, "Study leakage: train/val overlap"
assert len(train_studies & test_studies) == 0, "Study leakage: train/test overlap"
assert len(val_studies & test_studies) == 0, "Study leakage: val/test overlap"

def row_key(df: pd.DataFrame):
    return set((
        df["filepath"].astype(str) + "|" +
        df["liver_maskpath"].astype(str) + "|" +
        df["tumor_maskpath"].astype(str)
    ).tolist())

train_keys = row_key(splits["train"])
val_keys = row_key(splits["val"])
test_keys = row_key(splits["test"])

print("\nRow overlap:")
print(f"  train∩val : {len(train_keys & val_keys)}")
print(f"  train∩test: {len(train_keys & test_keys)}")
print(f"  val∩test  : {len(val_keys & test_keys)}")
assert len(train_keys & val_keys) == 0, "Row leakage: train/val overlap"
assert len(train_keys & test_keys) == 0, "Row leakage: train/test overlap"
assert len(val_keys & test_keys) == 0, "Row leakage: val/test overlap"

if PATHS["probe_csv"].exists():
    probe_df = pd.read_csv(PATHS["probe_csv"])
    probe_keys = row_key(probe_df)
    probe_overlap_ratio = len(probe_keys & test_keys) / max(1, len(test_keys))
    print(f"\nProbe/Test overlap ratio: {probe_overlap_ratio:.4f}")
    if probe_overlap_ratio > 0.95:
        print("Info: probe duplicates test for this dataset; excluded from training/reporting.")

### EDA: Class Imbalance and Data Health

In [ ]:
# Cell 3 — Split distribution and PNG integrity audit
train_df = splits["train"].copy()
val_df = splits["val"].copy()
test_df = splits["test"].copy()

# Auto-resolve actual PNG root on Kaggle (handles dataset_6 vs dataset_6/dataset_6)
def resolve_data_root(current_root: Path, df: pd.DataFrame) -> Path:
    sample_name = Path(str(df.iloc[0]["filepath"])).name
    candidates = [
        current_root,
        current_root / "dataset_6",
        current_root.parent / "dataset_6",
    ]
    for cand in candidates:
        if (cand / sample_name).exists():
            return cand
    return current_root

resolved_root = resolve_data_root(PATHS["data_root"], train_df)
if resolved_root != PATHS["data_root"]:
    print(f"Data root auto-corrected: {PATHS['data_root']} -> {resolved_root}")
    PATHS["data_root"] = resolved_root
else:
    print(f"Data root used: {PATHS['data_root']}")

for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    liver_pos = (~df["liver_mask_empty"].astype(bool)).mean() * 100
    tumor_pos = (~df["tumor_mask_empty"].astype(bool)).mean() * 100
    print(f"{split_name.upper():>5} | rows={len(df):,} | liver+={liver_pos:6.2f}% | tumor+={tumor_pos:6.2f}%")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(["train", "val", "test"], [len(train_df), len(val_df), len(test_df)], color=["#1f77b4", "#ff7f0e", "#2ca02c"])
ax[0].set_title("Split Size")
ax[0].set_ylabel("Slices")

imb = pd.DataFrame({
    "split": ["train", "val", "test"],
    "liver_positive_pct": [100 * (~train_df["liver_mask_empty"].astype(bool)).mean(),
                            100 * (~val_df["liver_mask_empty"].astype(bool)).mean(),
                            100 * (~test_df["liver_mask_empty"].astype(bool)).mean()],
    "tumor_positive_pct": [100 * (~train_df["tumor_mask_empty"].astype(bool)).mean(),
                            100 * (~val_df["tumor_mask_empty"].astype(bool)).mean(),
                            100 * (~test_df["tumor_mask_empty"].astype(bool)).mean()],
})
ax[1].plot(imb["split"], imb["liver_positive_pct"], marker="o", label="liver positive %")
ax[1].plot(imb["split"], imb["tumor_positive_pct"], marker="o", label="tumor positive %")
ax[1].set_title("Foreground Presence")
ax[1].set_ylim(0, 100)
ax[1].legend()
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

# PNG readability and mask-value audit (sample or full)
rng = np.random.default_rng(CFG["seed"])
if CFG["full_png_audit"]:
    sample_ids = np.arange(len(train_df))
else:
    audit_n = min(CFG["audit_sample_rows"], len(train_df))
    sample_ids = rng.choice(len(train_df), audit_n, replace=False)

missing = 0
unreadable = 0
shape_mismatch = 0
mask_values = []
intensity_means = []
liver_pos_px = []
tumor_pos_px = []

first_missing = None
for i in tqdm(sample_ids, desc="PNG audit"):
    row = train_df.iloc[int(i)]
    img_path = PATHS["data_root"] / Path(str(row["filepath"])).name
    liv_path = PATHS["data_root"] / Path(str(row["liver_maskpath"])).name
    tum_path = PATHS["data_root"] / Path(str(row["tumor_maskpath"])).name

    if not img_path.exists() or not liv_path.exists() or not tum_path.exists():
        missing += 1
        if first_missing is None:
            first_missing = (img_path, liv_path, tum_path)
        continue

    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    liv = cv2.imread(str(liv_path), cv2.IMREAD_GRAYSCALE)
    tum = cv2.imread(str(tum_path), cv2.IMREAD_GRAYSCALE)

    if img is None or liv is None or tum is None:
        unreadable += 1
        continue

    if img.shape != liv.shape or img.shape != tum.shape:
        shape_mismatch += 1

    liv_bin = (liv > 0).astype(np.uint8)
    tum_bin = (tum > 0).astype(np.uint8)

    intensity_means.append(float(img.mean()))
    mask_values.extend(np.unique(liv).tolist())
    mask_values.extend(np.unique(tum).tolist())
    liver_pos_px.append(float(liv_bin.mean()))
    tumor_pos_px.append(float(tum_bin.mean()))

audit_total = len(sample_ids)
print("\nAudit summary:")
print(f"  audited rows    : {audit_total}")
print(f"  missing triples : {missing}")
print(f"  unreadable      : {unreadable}")
print(f"  shape mismatches: {shape_mismatch}")
if intensity_means:
    print(f"  image mean(avg) : {np.mean(intensity_means):.2f}")
print(f"  mask unique vals: {sorted(set(mask_values))[:20]}")
if liver_pos_px:
    print(f"  liver pixel+ %  : {100*np.mean(liver_pos_px):.3f}")
if tumor_pos_px:
    print(f"  tumor pixel+ %  : {100*np.mean(tumor_pos_px):.4f}")
if first_missing is not None:
    print("  first missing example:")
    print(f"    image: {first_missing[0]}")
    print(f"    liver: {first_missing[1]}")
    print(f"    tumor: {first_missing[2]}")

assert missing == 0, "Missing PNG triples found in audited set. Check DATA_ROOT_OVERRIDE/CSV_ROOT_OVERRIDE and rerun Cell 2."
assert unreadable == 0, "Unreadable PNG triples found in audited set"
assert shape_mismatch == 0, "Image/mask shape mismatch found in audited set"

## 2) Strict Dataset, Preprocessing, and Dataloaders

In [ ]:
# Cell 4 — Strict path resolution, preprocessing, augmentations, and dataset class
class CTPreprocessor:
    def __init__(
        self,
        image_size=256,
        clahe_clip=2.0,
        clahe_grid=(8, 8),
        use_clahe=True,
        apply_median=False,
    ):
        self.image_size = image_size
        self.use_clahe = use_clahe
        self.apply_median = apply_median
        self.clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=clahe_grid)

    def __call__(self, image_u8: np.ndarray) -> np.ndarray:
        x = image_u8.copy()
        # robust percentile clip for CT-like PNG dynamic range stabilization
        lo, hi = np.percentile(x, 0.5), np.percentile(x, 99.5)
        if hi > lo:
            x = np.clip(x, lo, hi)
            x = ((x - lo) / (hi - lo + 1e-8) * 255.0).astype(np.uint8)
        if self.apply_median:
            x = cv2.medianBlur(x, 3)
        if self.use_clahe:
            x = self.clahe.apply(x)
        x = cv2.resize(x, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        return x

def resolve_paths(row: pd.Series, root: Path):
    return (
        root / Path(str(row["filepath"])).name,
        root / Path(str(row["liver_maskpath"])).name,
        root / Path(str(row["tumor_maskpath"])).name,
    )

def augment_pair(img, liv, tum):
    if np.random.rand() < 0.5:
        img, liv, tum = np.fliplr(img).copy(), np.fliplr(liv).copy(), np.fliplr(tum).copy()
    if np.random.rand() < 0.2:
        img, liv, tum = np.flipud(img).copy(), np.flipud(liv).copy(), np.flipud(tum).copy()
    if np.random.rand() < 0.6:
        k = np.random.randint(0, 4)
        img, liv, tum = np.rot90(img, k).copy(), np.rot90(liv, k).copy(), np.rot90(tum, k).copy()
    if np.random.rand() < 0.5:
        factor = np.random.uniform(0.9, 1.1)
        img = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)
    return img, liv, tum

class LITSDatasetStrict(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        root_dir: Path,
        image_size=256,
        augment=False,
        use_clahe=True,
        apply_median=False,
    ):
        self.df = df.reset_index(drop=True).copy()
        self.root_dir = Path(root_dir)
        self.image_size = image_size
        self.augment = augment
        self.preprocess = CTPreprocessor(
            image_size=image_size,
            use_clahe=use_clahe,
            apply_median=apply_median,
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path, liv_path, tum_path = resolve_paths(row, self.root_dir)

        for p in [img_path, liv_path, tum_path]:
            if not p.exists():
                raise FileNotFoundError(f"Missing file: {p}")

        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        liv = cv2.imread(str(liv_path), cv2.IMREAD_GRAYSCALE)
        tum = cv2.imread(str(tum_path), cv2.IMREAD_GRAYSCALE)
        if img is None or liv is None or tum is None:
            raise ValueError(f"Unreadable PNG triple at idx={idx}")

        img = self.preprocess(img)
        liv = cv2.resize(liv, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
        tum = cv2.resize(tum, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)

        liv = (liv > 0).astype(np.float32)
        tum = (tum > 0).astype(np.float32)

        if self.augment:
            img, liv, tum = augment_pair(img, liv, tum)

        x = torch.from_numpy(img.astype(np.float32) / 255.0).unsqueeze(0)
        y = torch.from_numpy(np.stack([liv, tum], axis=0).astype(np.float32))
        return x, y

# Preprocessing policy lock from audit statistics
mean_tumor_px = float(np.mean(tumor_pos_px)) if len(tumor_pos_px) > 0 else 0.0
use_clahe_policy = True
apply_median_policy = mean_tumor_px < 0.003
print(f"Preprocess policy | CLAHE={use_clahe_policy} | median={apply_median_policy} | tumor_px_mean={mean_tumor_px:.6f}")

# Build datasets
train_ds = LITSDatasetStrict(
    train_df,
    PATHS["data_root"],
    image_size=CFG["image_size"],
    augment=True,
    use_clahe=use_clahe_policy,
    apply_median=apply_median_policy,
)
val_ds = LITSDatasetStrict(
    val_df,
    PATHS["data_root"],
    image_size=CFG["image_size"],
    augment=False,
    use_clahe=use_clahe_policy,
    apply_median=apply_median_policy,
)
test_ds = LITSDatasetStrict(
    test_df,
    PATHS["data_root"],
    image_size=CFG["image_size"],
    augment=False,
    use_clahe=use_clahe_policy,
    apply_median=apply_median_policy,
)

# Weighted sampler from CSV metadata (fast, no full mask scan)
train_weight = np.ones(len(train_df), dtype=np.float32)
train_weight += (~train_df["liver_mask_empty"].astype(bool)).values.astype(np.float32) * (CFG["sampler_liver_boost"] - 1.0)
train_weight += (~train_df["tumor_mask_empty"].astype(bool)).values.astype(np.float32) * (CFG["sampler_tumor_boost"] - 1.0)
sampler = WeightedRandomSampler(weights=train_weight.tolist(), num_samples=len(train_weight), replacement=True)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    sampler=sampler,
    num_workers=CFG["num_workers"],
    pin_memory=CFG["pin_memory"],
    persistent_workers=CFG["num_workers"] > 0,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=CFG["pin_memory"],
    persistent_workers=CFG["num_workers"] > 0,
)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=CFG["pin_memory"],
    persistent_workers=CFG["num_workers"] > 0,
)

print(f"train batches: {len(train_loader)} | val batches: {len(val_loader)} | test batches: {len(test_loader)}")

## 3) Model, Tumor-Focused Loss, and Metrics

In [ ]:
# Cell 5 — Model, losses, and metrics aligned to tumor-first objective
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=2, feat=(64, 128, 256, 512)):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()

        ch = in_ch
        for f in feat:
            self.downs.append(DoubleConv(ch, f))
            ch = f
        self.bottleneck = DoubleConv(feat[-1], feat[-1] * 2)

        rev = list(reversed(feat))
        up_in = feat[-1] * 2
        for f in rev:
            self.ups.append(nn.ConvTranspose2d(up_in, f, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(f * 2, f))
            up_in = f

        self.head = nn.Conv2d(feat[0], out_ch, 1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skips = skips[::-1]

        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skips[i // 2]
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = self.ups[i + 1](torch.cat([skip, x], dim=1))
        return self.head(x)

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).float()
        targets = targets.float()
        inter = (probs * targets).sum(dim=(2, 3))
        denom = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        dice = (2.0 * inter + self.smooth) / (denom + self.smooth)
        return 1.0 - dice.mean()

class TumorFocusedCombinedLoss(nn.Module):
    def __init__(self, tumor_pos_weight=3.0, smooth=1e-6, outside_liver_penalty=0.0):
        super().__init__()
        pos_w = torch.tensor([1.0, float(tumor_pos_weight)], dtype=torch.float32)
        self.register_buffer("pos_w", pos_w)
        self.dice = DiceLoss(smooth=smooth)
        self.outside_liver_penalty = float(outside_liver_penalty)

    def forward(self, logits, targets):
        logits = logits.float()
        targets = targets.float()
        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none",
        )
        w = self.pos_w.view(1, 2, 1, 1)
        bce = (bce * w).mean()

        probs = torch.sigmoid(logits)
        liver_prob = probs[:, 0]
        tumor_prob = probs[:, 1]
        # Penalize predicted tumor in low-liver regions to control false tumor blobs
        outside_region = (1.0 - liver_prob).detach()
        outside_loss = (tumor_prob * outside_region).mean()

        return 0.5 * bce + 0.5 * self.dice(logits, targets) + self.outside_liver_penalty * outside_loss

@torch.no_grad()
def dice_per_channel(logits, targets, th_liver=0.5, th_tumor=0.5, smooth=1e-6):
    probs = torch.sigmoid(logits).float()
    preds = torch.stack([
        (probs[:, 0] > th_liver).float(),
        (probs[:, 1] > th_tumor).float(),
    ], dim=1)
    targets = targets.float()

    out = {}
    for ch, name in enumerate(["liver", "tumor"]):
        p = preds[:, ch].reshape(preds.shape[0], -1)
        t = targets[:, ch].reshape(targets.shape[0], -1)
        inter = (p * t).sum(dim=1)
        denom = p.sum(dim=1) + t.sum(dim=1)
        all_slices = ((2 * inter + smooth) / (denom + smooth)).mean().item()
        fg_mask = (t.sum(dim=1) > 0)
        fg_only = ((2 * inter[fg_mask] + smooth) / (denom[fg_mask] + smooth)).mean().item() if fg_mask.any() else float("nan")
        out[f"{name}_dice_all"] = all_slices
        out[f"{name}_dice_fg"] = fg_only
    out["mean_dice_all"] = np.nanmean([out["liver_dice_all"], out["tumor_dice_all"]])
    out["mean_dice_fg"] = np.nanmean([out["liver_dice_fg"], out["tumor_dice_fg"]])
    return out

model = UNet(in_ch=1, out_ch=2).to(CFG["device"])
criterion = TumorFocusedCombinedLoss(
    CFG["tumor_pos_weight"],
    smooth=CFG["dice_smooth"],
    outside_liver_penalty=CFG.get("outside_liver_penalty", 0.0),
).to(CFG["device"])
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4) Unattended Training Pipeline (Kaggle Safe Mode)

In [ ]:
# Cell 6 — Unattended-safe train/validate loops + resilient checkpointing
optimizer = optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=CFG["lr_factor"], patience=CFG["lr_patience"]
)
scaler = torch.amp.GradScaler("cuda", enabled=(CFG["device"] == "cuda"))

best_ckpt_path = Path(CFG["model_dir"]) / "best_model_tumor_dice.pth"
latest_ckpt_path = Path(CFG["model_dir"]) / "latest_model.pth"
log_path = Path(CFG["save_dir"]) / "training_log.csv"
state_path = Path(CFG["save_dir"]) / "training_state.json"
status_path = Path(CFG["save_dir"]) / "live_status.json"

history = []
best_tumor_dice = -1.0
best_clinical_score = -1.0
patience_counter = 0
bad_epoch_counter = 0
start_epoch = 1

@torch.no_grad()
def run_eval(model, loader, criterion, th_liver=0.5, th_tumor=0.5):
    model.eval()
    losses = []
    agg = {"liver_dice_all": [], "tumor_dice_all": [], "mean_dice_all": [], "liver_dice_fg": [], "tumor_dice_fg": [], "mean_dice_fg": []}
    tumor_fp = 0.0
    tumor_tn = 0.0
    tumor_outside_sum = 0.0
    tumor_pred_sum = 0.0

    for x, y in tqdm(loader, leave=False):
        x = x.to(CFG["device"], non_blocking=True)
        y = y.to(CFG["device"], non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(CFG["device"] == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)
        if not torch.isfinite(loss):
            continue
        losses.append(loss.item())

        m = dice_per_channel(logits, y, th_liver=th_liver, th_tumor=th_tumor, smooth=CFG["dice_smooth"])
        for k in agg:
            agg[k].append(m[k])

        probs = torch.sigmoid(logits.float())
        pred_l = (probs[:, 0] > th_liver).float()
        pred_t = (probs[:, 1] > th_tumor).float()
        true_t = y[:, 1].float()
        tumor_fp += float((pred_t * (1.0 - true_t)).sum().item())
        tumor_tn += float(((1.0 - pred_t) * (1.0 - true_t)).sum().item())
        tumor_outside_sum += float((pred_t * (1.0 - pred_l)).sum().item())
        tumor_pred_sum += float(pred_t.sum().item())

    out = {k: float(np.nanmean(v)) if len(v) > 0 else float("nan") for k, v in agg.items()}
    out["loss"] = float(np.mean(losses)) if losses else float("nan")
    out["tumor_fp_rate"] = tumor_fp / max(1.0, (tumor_fp + tumor_tn))
    out["tumor_outside_liver_rate"] = tumor_outside_sum / max(1.0, tumor_pred_sum)

    w = CFG["composite_weights"]
    out["clinical_score"] = (
        w["tumor"] * out["tumor_dice_all"]
        + w["liver"] * out["liver_dice_all"]
        - w["fp_penalty"] * out["tumor_fp_rate"]
        - w.get("outside_penalty", 0.0) * out["tumor_outside_liver_rate"]
    )
    return out

def save_checkpoint_atomic(payload, path: Path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)

def save_training_state(last_row=None, run_hours=0.0):
    state_obj = {
        "start_epoch_next": start_epoch,
        "best_tumor_dice": best_tumor_dice,
        "best_clinical_score": best_clinical_score,
        "patience_counter": patience_counter,
        "bad_epoch_counter": bad_epoch_counter,
        "latest_ckpt": str(latest_ckpt_path),
        "best_ckpt": str(best_ckpt_path),
        "run_hours": run_hours,
    }
    with open(state_path, "w", encoding="utf-8") as f:
        json.dump(state_obj, f, indent=2)

    if last_row is not None:
        live_obj = {
            "preset": PRESET_NAME,
            "epoch": int(last_row["epoch"]),
            "train_loss": float(last_row["train_loss"]),
            "val_loss": float(last_row["val_loss"]),
            "val_tumor_dice_all": float(last_row["val_tumor_dice_all"]),
            "val_liver_dice_all": float(last_row["val_liver_dice_all"]),
            "val_tumor_fp_rate": float(last_row["val_tumor_fp_rate"]),
            "val_clinical_score": float(last_row["val_clinical_score"]),
            "best_tumor_dice": float(best_tumor_dice),
            "best_clinical_score": float(best_clinical_score),
            "run_hours": float(run_hours),
            "max_run_hours": float(CFG["max_run_hours"]),
            "remaining_hours": float(max(0.0, CFG["max_run_hours"] - run_hours)),
        }
        with open(status_path, "w", encoding="utf-8") as f:
            json.dump(live_obj, f, indent=2)

# Auto resume if available
if CFG["auto_resume"] and latest_ckpt_path.exists():
    ckpt = torch.load(latest_ckpt_path, map_location=CFG["device"], weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch = int(ckpt["epoch"]) + 1
    best_tumor_dice = float(ckpt.get("best_tumor_dice", -1.0))
    best_clinical_score = float(ckpt.get("best_clinical_score", -1.0))
    if log_path.exists():
        history = pd.read_csv(log_path).to_dict(orient="records")
    print(f"Auto-resume from epoch {start_epoch}")

# Preflight smoke test on small number of steps
model.train()
optimizer.zero_grad(set_to_none=True)
smoke_ok = True
for step, (x, y) in enumerate(train_loader, start=1):
    if step > CFG["smoke_train_steps"]:
        break
    x = x.to(CFG["device"], non_blocking=True)
    y = y.to(CFG["device"], non_blocking=True)
    with torch.amp.autocast("cuda", enabled=(CFG["device"] == "cuda")):
        logits = model(x)
        loss = criterion(logits, y)
    if not torch.isfinite(loss):
        smoke_ok = False
        break
    (loss / CFG["accum_steps"]).backward()
    if step % CFG["accum_steps"] == 0:
        optimizer.zero_grad(set_to_none=True)
if not smoke_ok:
    raise RuntimeError("Smoke test failed: non-finite loss in preflight checks")
optimizer.zero_grad(set_to_none=True)
print("Smoke test passed.")

run_start_time = time.time()
nan_batch_count = 0
for epoch in range(start_epoch, CFG["epochs"] + 1):
    elapsed_hours = (time.time() - run_start_time) / 3600.0
    if elapsed_hours >= CFG["max_run_hours"]:
        print(f"Reached max_run_hours={CFG['max_run_hours']}. Stopping safely for resume.")
        save_training_state(run_hours=elapsed_hours)
        break

    start_epoch = epoch + 1
    model.train()
    t0 = time.time()
    train_losses = []
    train_liver, train_tumor = [], []

    if epoch <= CFG["warmup_epochs"]:
        warmup_scale = 0.2 + 0.8 * (epoch / CFG["warmup_epochs"])
        for pg in optimizer.param_groups:
            pg["lr"] = CFG["lr"] * warmup_scale

    optimizer.zero_grad(set_to_none=True)
    for step, (x, y) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch:02d} train", leave=False), start=1):
        x = x.to(CFG["device"], non_blocking=True)
        y = y.to(CFG["device"], non_blocking=True)

        with torch.amp.autocast("cuda", enabled=(CFG["device"] == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        if not torch.isfinite(loss):
            nan_batch_count += 1
            optimizer.zero_grad(set_to_none=True)
            if nan_batch_count >= CFG["nan_loss_abort_batches"]:
                raise RuntimeError("Too many non-finite batches; aborting unattended run")
            continue

        loss_scaled = loss / CFG["accum_steps"]
        scaler.scale(loss_scaled).backward()

        if step % CFG["accum_steps"] == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        train_losses.append(loss.item())
        dm = dice_per_channel(logits, y)
        train_liver.append(dm["liver_dice_all"])
        train_tumor.append(dm["tumor_dice_all"])

    if epoch % CFG["eval_every_n_epochs"] != 0:
        continue

    val_stats = run_eval(model, val_loader, criterion, th_liver=0.5, th_tumor=0.5)
    if not np.isfinite(val_stats["clinical_score"]):
        bad_epoch_counter += 1
        save_training_state(run_hours=(time.time() - run_start_time) / 3600.0)
        if bad_epoch_counter >= CFG["max_bad_epochs"]:
            raise RuntimeError("Repeated invalid validation epochs; stopping run")
        continue

    bad_epoch_counter = 0
    scheduler.step(val_stats["tumor_dice_all"])

    row = {
        "epoch": epoch,
        "lr": optimizer.param_groups[0]["lr"],
        "train_loss": float(np.mean(train_losses)) if len(train_losses) else float("nan"),
        "train_liver_dice_all": float(np.mean(train_liver)) if len(train_liver) else float("nan"),
        "train_tumor_dice_all": float(np.mean(train_tumor)) if len(train_tumor) else float("nan"),
        "val_loss": val_stats["loss"],
        "val_liver_dice_all": val_stats["liver_dice_all"],
        "val_tumor_dice_all": val_stats["tumor_dice_all"],
        "val_mean_dice_all": val_stats["mean_dice_all"],
        "val_liver_dice_fg": val_stats["liver_dice_fg"],
        "val_tumor_dice_fg": val_stats["tumor_dice_fg"],
        "val_mean_dice_fg": val_stats["mean_dice_fg"],
        "val_tumor_fp_rate": val_stats["tumor_fp_rate"],
        "val_tumor_outside_liver_rate": val_stats["tumor_outside_liver_rate"],
        "val_clinical_score": val_stats["clinical_score"],
        "epoch_time_s": time.time() - t0,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(log_path, index=False)

    if epoch % CFG["save_every_n_epochs"] == 0:
        save_checkpoint_atomic({
            "epoch": epoch,
            "config": CFG,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_tumor_dice": best_tumor_dice,
            "best_clinical_score": best_clinical_score,
        }, latest_ckpt_path)

    meets_guards = (
        row["val_liver_dice_all"] >= CFG["liver_min_dice_guard"]
        and row["val_tumor_fp_rate"] <= CFG["max_tumor_fp_rate"]
        and row["val_tumor_outside_liver_rate"] <= CFG.get("max_tumor_outside_liver_rate", 1.0)
    )

    improved = False
    if meets_guards:
        improved = row["val_clinical_score"] > best_clinical_score
    else:
        if best_clinical_score < 0 and row["val_tumor_dice_all"] > best_tumor_dice:
            improved = True

    if improved:
        best_tumor_dice = max(best_tumor_dice, row["val_tumor_dice_all"])
        best_clinical_score = row["val_clinical_score"]
        patience_counter = 0
        save_checkpoint_atomic({
            "epoch": epoch,
            "config": CFG,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_tumor_dice": best_tumor_dice,
            "best_clinical_score": best_clinical_score,
            "best_row": row,
        }, best_ckpt_path)
    else:
        patience_counter += 1

    current_hours = (time.time() - run_start_time) / 3600.0
    save_training_state(last_row=row, run_hours=current_hours)

    print(
        f"Epoch {epoch:03d} | train_loss={row['train_loss']:.4f} | val_loss={row['val_loss']:.4f} | "
        f"tumor={row['val_tumor_dice_all']:.4f} | liver={row['val_liver_dice_all']:.4f} | "
        f"fp_rate={row['val_tumor_fp_rate']:.5f} | out_liver={row['val_tumor_outside_liver_rate']:.4f} | score={row['val_clinical_score']:.4f} | "
        f"guards={'OK' if meets_guards else 'NO'} | no_improve={patience_counter}/{CFG['early_stop_patience']} | "
        f"elapsed={current_hours:.2f}h/{CFG['max_run_hours']:.1f}h"
    )

    if patience_counter >= CFG["early_stop_patience"]:
        print("Early stopping triggered.")
        break

print(f"Best checkpoint: {best_ckpt_path}")
print(f"Best clinical score: {best_clinical_score:.4f}")
print(f"Best tumor dice: {best_tumor_dice:.4f}")
print(f"Latest checkpoint: {latest_ckpt_path}")
print(f"Training log: {log_path}")

## 5) Training Curves and Learning Diagnostics

In [ ]:
log_df = pd.read_csv(log_path)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(log_df["epoch"], log_df["train_loss"], label="train loss")
axes[0].plot(log_df["epoch"], log_df["val_loss"], label="val loss")
axes[0].set_title("Loss")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(log_df["epoch"], log_df["val_liver_dice_all"], label="val liver dice")
axes[1].plot(log_df["epoch"], log_df["val_liver_dice_fg"], label="val liver dice fg")
axes[1].set_title("Liver Dice")
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)
axes[1].legend()

axes[2].plot(log_df["epoch"], log_df["val_tumor_dice_all"], label="val tumor dice")
axes[2].plot(log_df["epoch"], log_df["val_tumor_dice_fg"], label="val tumor dice fg")
axes[2].set_title("Tumor Dice (Primary)")
axes[2].set_ylim(0, 1)
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.savefig(Path(CFG["save_dir"]) / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

best_idx = log_df["val_tumor_dice_all"].idxmax()
best_row = log_df.loc[best_idx]
print("Best epoch by val_tumor_dice_all:")
print(best_row[["epoch", "val_tumor_dice_all", "val_liver_dice_all", "val_loss"]])

## 6) Threshold Sweep on Validation (Tumor First)

In [ ]:
best_ckpt = torch.load(best_ckpt_path, map_location=CFG["device"], weights_only=False)
model.load_state_dict(best_ckpt["model_state_dict"])
model.eval()

threshold_grid = np.linspace(0.20, 0.80, 13)
rows = []

for th_t in threshold_grid:
    st = run_eval(model, val_loader, criterion, th_liver=0.5, th_tumor=float(th_t))
    rows.append({"th_tumor": float(th_t), **st})

th_df = pd.DataFrame(rows).sort_values("clinical_score", ascending=False).reset_index(drop=True)
best_th_tumor = float(th_df.iloc[0]["th_tumor"])

# optional liver threshold sweep around 0.5
liver_rows = []
for th_l in np.linspace(0.35, 0.65, 7):
    st = run_eval(model, val_loader, criterion, th_liver=float(th_l), th_tumor=best_th_tumor)
    liver_rows.append({"th_liver": float(th_l), **st})
liver_df = pd.DataFrame(liver_rows).sort_values("clinical_score", ascending=False).reset_index(drop=True)
best_th_liver = float(liver_df.iloc[0]["th_liver"])

th_df.to_csv(Path(CFG["save_dir"]) / "val_tumor_threshold_sweep.csv", index=False)
liver_df.to_csv(Path(CFG["save_dir"]) / "val_liver_threshold_sweep.csv", index=False)

print(f"Best tumor threshold: {best_th_tumor:.3f}")
print(f"Best liver threshold: {best_th_liver:.3f}")
print(th_df.head(5)[["th_tumor", "tumor_dice_all", "liver_dice_all", "tumor_fp_rate", "clinical_score"]])

## 7) Held-Out Test Evaluation (Final Reportable Metrics)

In [ ]:
@torch.no_grad()
def confusion_metrics(model, loader, th_liver, th_tumor):
    model.eval()
    tp = np.zeros(2, dtype=np.float64)
    fp = np.zeros(2, dtype=np.float64)
    fn = np.zeros(2, dtype=np.float64)
    tn = np.zeros(2, dtype=np.float64)

    for x, y in tqdm(loader, desc="Eval", leave=False):
        x = x.to(CFG["device"], non_blocking=True)
        y = y.to(CFG["device"], non_blocking=True).float()
        logits = model(x)
        probs = torch.sigmoid(logits)
        pred = torch.stack([
            (probs[:, 0] > th_liver).float(),
            (probs[:, 1] > th_tumor).float(),
        ], dim=1)

        for ch in [0, 1]:
            p = pred[:, ch].reshape(-1).cpu().numpy()
            t = y[:, ch].reshape(-1).cpu().numpy()
            tp[ch] += float((p * t).sum())
            fp[ch] += float((p * (1 - t)).sum())
            fn[ch] += float(((1 - p) * t).sum())
            tn[ch] += float(((1 - p) * (1 - t)).sum())

    def pack(ch):
        smooth = CFG["dice_smooth"]
        dice = (2 * tp[ch] + smooth) / (2 * tp[ch] + fp[ch] + fn[ch] + smooth)
        iou = (tp[ch] + smooth) / (tp[ch] + fp[ch] + fn[ch] + smooth)
        precision = (tp[ch] + smooth) / (tp[ch] + fp[ch] + smooth)
        recall = (tp[ch] + smooth) / (tp[ch] + fn[ch] + smooth)
        f1 = (2 * precision * recall) / (precision + recall + smooth)
        acc = (tp[ch] + tn[ch]) / (tp[ch] + fp[ch] + fn[ch] + tn[ch] + smooth)
        return dict(dice=dice, iou=iou, precision=precision, recall=recall, f1=f1, accuracy=acc,
                    tp=tp[ch], fp=fp[ch], fn=fn[ch], tn=tn[ch])

    return {"liver": pack(0), "tumor": pack(1)}

val_metrics = confusion_metrics(model, val_loader, best_th_liver, best_th_tumor)
test_metrics = confusion_metrics(model, test_loader, best_th_liver, best_th_tumor)

report_rows = []
for split_name, met in [("val", val_metrics), ("test", test_metrics)]:
    for ch in ["liver", "tumor"]:
        row = {"split": split_name, "channel": ch}
        row.update(met[ch])
        report_rows.append(row)

report_df = pd.DataFrame(report_rows)
report_df.to_csv(Path(CFG["save_dir"]) / "final_metrics_val_test.csv", index=False)
print(report_df[["split", "channel", "dice", "iou", "precision", "recall", "f1"]])
print(f"Saved: {Path(CFG['save_dir']) / 'final_metrics_val_test.csv'}")

## 8) Qualitative Predictions and Report Artifacts

In [ ]:
# Sample qualitative predictions from test set
rng = np.random.default_rng(CFG["seed"])
indices = rng.choice(len(test_ds), min(8, len(test_ds)), replace=False)

fig, axes = plt.subplots(len(indices), 5, figsize=(18, 3.3 * len(indices)))
if len(indices) == 1:
    axes = np.expand_dims(axes, axis=0)

for r, idx in enumerate(indices):
    x, y = test_ds[int(idx)]
    with torch.no_grad():
        logits = model(x.unsqueeze(0).to(CFG["device"]))
        probs = torch.sigmoid(logits).squeeze(0).cpu()
    pred_l = (probs[0] > best_th_liver).numpy().astype(np.float32)
    pred_t = (probs[1] > best_th_tumor).numpy().astype(np.float32)

    img = x[0].numpy()
    liv_gt = y[0].numpy()
    tum_gt = y[1].numpy()

    axes[r, 0].imshow(img, cmap="gray")
    axes[r, 0].set_title(f"CT idx={idx}")
    axes[r, 1].imshow(liv_gt, cmap="Greens", vmin=0, vmax=1)
    axes[r, 1].set_title("Liver GT")
    axes[r, 2].imshow(pred_l, cmap="Greens", vmin=0, vmax=1)
    axes[r, 2].set_title("Liver Pred")
    axes[r, 3].imshow(tum_gt, cmap="Reds", vmin=0, vmax=1)
    axes[r, 3].set_title("Tumor GT")
    axes[r, 4].imshow(pred_t, cmap="Reds", vmin=0, vmax=1)
    axes[r, 4].set_title("Tumor Pred")
    for c in range(5):
        axes[r, c].axis("off")

plt.tight_layout()
qual_path = Path(CFG["save_dir"]) / "qualitative_test_predictions.png"
plt.savefig(qual_path, dpi=150, bbox_inches="tight")
plt.show()

summary = {
    "run_name": CFG["run_name"],
    "best_tumor_threshold": best_th_tumor,
    "best_liver_threshold": best_th_liver,
    "best_checkpoint": str(best_ckpt_path),
    "device": CFG["device"],
    "val_tumor_dice": float(report_df[(report_df.split=="val") & (report_df.channel=="tumor")]["dice"].iloc[0]),
    "test_tumor_dice": float(report_df[(report_df.split=="test") & (report_df.channel=="tumor")]["dice"].iloc[0]),
}
with open(Path(CFG["save_dir"]) / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved qualitative predictions: {qual_path}")
print(f"Saved run summary: {Path(CFG['save_dir']) / 'run_summary.json'}")

## Notes for Best-Accuracy Practice

- Keep `Tumor Dice` as primary model-selection metric.
- Track both all-slice and foreground-only Dice to avoid inflated scores from empty masks.
- Keep split protocol fixed and never tune on test set.
- For better tumor performance after baseline convergence, run controlled experiments with:
  - focal-tversky variant (tumor channel)
  - threshold tuning range expansion
  - light test-time augmentation
  - liver-ROI-constrained tumor refinement

## 9) One-Click Execution Checklist

Run cells in order:
1. Environment/config
2. Data audit + EDA
3. Dataset/model definitions
4. Unattended-safe training loop (includes smoke test + auto-resume)
5. Curves
6. Threshold calibration
7. Final val/test evaluation
8. Qualitative predictions + summary export

Unattended safety features:
- auto-resume from `models/latest_model.pth`
- atomic checkpoint writes (`.tmp` -> rename)
- smoke-test before long run
- non-finite loss guards with abort thresholds
- max runtime guard per session (`CFG["max_run_hours"]`, default `11.5`)
- persistent training state in `results/training_state.json`
- live progress snapshot in `results/live_status.json`

Kaggle budget guidance:
- daily session budget: keep `max_run_hours=11.5`
- weekly budget: target up to `CFG["weekly_gpu_budget_hours"]` (default `30`)
- use auto-resume to continue next day without losing progress

Primary artifact files:
- `models/best_model_tumor_dice.pth`
- `models/latest_model.pth`
- `results/training_log.csv`
- `results/training_state.json`
- `results/live_status.json`
- `results/val_tumor_threshold_sweep.csv`
- `results/final_metrics_val_test.csv`
- `results/run_summary.json`

In [ ]:
# Final report table + continue/stop training guidance
display_cols = ["split", "channel", "dice", "iou", "precision", "recall", "f1", "accuracy", "fp", "fn"]
report_pivot = report_df[display_cols].copy()
report_pivot.to_csv(Path(CFG["save_dir"]) / "report_table.csv", index=False)
print(report_pivot)

# Save config used for this run
with open(Path(CFG["save_dir"]) / "run_config.json", "w", encoding="utf-8") as f:
    json.dump({k: (float(v) if isinstance(v, np.floating) else v) for k, v in CFG.items()}, f, indent=2)

# Practical training decision guidance
tumor_val = float(report_df[(report_df.split == "val") & (report_df.channel == "tumor")]["dice"].iloc[0])
tumor_test = float(report_df[(report_df.split == "test") & (report_df.channel == "tumor")]["dice"].iloc[0])
val_test_gap = tumor_val - tumor_test

test_tumor_fp_rate = float(report_df[(report_df.split == "test") & (report_df.channel == "tumor")]["fp"].iloc[0]) / max(
    1.0,
    float(report_df[(report_df.split == "test") & (report_df.channel == "tumor")]["fp"].iloc[0]) +
    float(report_df[(report_df.split == "test") & (report_df.channel == "tumor")]["tn"].iloc[0]),
)

if tumor_val < 0.55:
    decision = "CONTINUE: model underfitting for tumor; run Stage-B finetuning and/or stronger tumor loss."
elif val_test_gap > 0.06:
    decision = "STOP and REGULARIZE: overfitting likely; keep best checkpoint, tune augment/loss, then retrain."
else:
    decision = "STOP: good generalization; use this checkpoint for final reporting and inference."

test_liver_dice = float(report_df[(report_df.split == "test") & (report_df.channel == "liver")]["dice"].iloc[0])
val_liver_dice = float(report_df[(report_df.split == "val") & (report_df.channel == "liver")]["dice"].iloc[0])

# tumor outside liver proxy from confusion table is not directly available; we use val tracked metric if present
val_outside_proxy = float(log_df["val_tumor_outside_liver_rate"].dropna().iloc[-1]) if "val_tumor_outside_liver_rate" in log_df.columns and len(log_df) > 0 else 0.0

ready_for_aggressive = (
    PRESET_NAME == "balanced_best"
    and tumor_val >= STAGE_GATE["min_val_tumor_dice"]
    and tumor_test >= STAGE_GATE["min_test_tumor_dice"]
    and val_liver_dice >= STAGE_GATE["min_val_liver_dice"]
    and test_liver_dice >= STAGE_GATE["min_test_liver_dice"]
    and val_test_gap <= STAGE_GATE["max_val_test_tumor_gap"]
    and test_tumor_fp_rate <= STAGE_GATE["max_test_tumor_fp_rate"]
    and val_outside_proxy <= STAGE_GATE["max_test_tumor_outside_liver_rate"]
)

if ready_for_aggressive:
    stage_msg = "PROMOTE: Balanced goal achieved. Switch PRESET_NAME to aggressive_max_accuracy."
else:
    stage_msg = "HOLD: Continue balanced_best until gate metrics are met."

with open(Path(CFG["save_dir"]) / "training_decision.txt", "w", encoding="utf-8") as f:
    f.write(decision + "\n")
    f.write(stage_msg + "\n")

print("Notebook pipeline complete.")
print(f"Decision: {decision}")
print(f"Stage gate: {stage_msg}")
print(f"Artifacts in: {CFG['save_dir']} and {CFG['model_dir']}")

## 10) Long-Run Optimization Protocol (Optional but Recommended)

Use this schedule for long Kaggle sessions:
- Stage A: baseline training with current `CFG`
- Stage B: fine-tune from best checkpoint at lower LR
- Stage C: threshold re-calibration + optional light TTA

Run Stage B only if Stage A has plateaued and still shows potential on validation tumor Dice.

In [ ]:
# Optional Stage-B fine-tuning and light TTA helpers

def load_best_for_finetune(model, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=CFG["device"], weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    return ckpt

@torch.no_grad()
def predict_with_tta(model, x):
    # light TTA: identity + horizontal flip
    p0 = torch.sigmoid(model(x))
    xf = torch.flip(x, dims=[3])
    pf = torch.sigmoid(model(xf))
    pf = torch.flip(pf, dims=[3])
    return 0.5 * (p0 + pf)

# Example finetune config (manual run)
FINETUNE_CFG = {
    "epochs": 25,
    "lr": 8e-5,
    "early_stop_patience": 8,
}

print("Finetune helper ready.")
print("If Stage-A plateaus, reload best checkpoint and run extra epochs with lower LR.")

## Stage Strategy

Use this progression:
1. Run `balanced_best` until stage gate says `PROMOTE`.
2. Then switch `PRESET_NAME` to `aggressive_max_accuracy`.
3. Keep auto-resume on and continue from latest checkpoint.

This prevents wasting weekly GPU hours on aggressive tuning before base generalization is stable.